# Soal Peminatan CV

Halo semua, selamat datang di soal soal CV

Ga bakal sulit kok

jadi disini ada 1 folder zip "gambars", dimana isinya ya gambar. tugas kalian adalah:

1. pisahkan objeck foreground dari backgroundnya
2. pindahkan hanya objek foreground-nya ke folder "result"
3. bebas pake cara apa aja okay
4. kasih alesan ya kenapa pake cara kayak gitu, terus analisa kenapa hasilnya begitu 

## Dokumentasi

untuk proses pengerjaan, bebas kalian mau bikin fungsi sendiri, literally from scratch atau menggunakan library cv2. yang sebenernya bakal lebih berguna karena digunakan di industri kan. ditambah waktu kalian terbatas

### Load gambar
untuk proses loading gambar didalam folder, ada beberapa cara bisa menggunakan library os atau glob atau pathlib. bebas kalian pake yang mana

misal kalau pake glob sesimple
```
path = glob.glob("path/*.ekstensi_file")
```
misal extensionnya beda beda, bisa kalian bikinin list yang isinya semua extension yang ada dulu

```
extension = ["png", "jpg", "jpeg"]
```
terus yaudah tinggal pake
```
paths = [glob.glob(f'/content/gambar/gambars/*.{ext}') for ext in extension]
```

tapi glob itu balikin list isinya path, kalau kalian ikutin cara tadi, hasilnya akan nested list. bisa pake "extend" biar listnya digabung

```
image_paths = [output_path]
for ext in extension:
    image_paths.extend(glob.glob(f'{path_dataset}*.{ext}'))
```


In [7]:
!unzip -o gambars.zip -d gambar

Archive:  gambars.zip
   creating: gambar/gambars/
  inflating: gambar/gambars/dihitamkan.jpeg  
  inflating: gambar/gambars/goblinCOC.png  
  inflating: gambar/gambars/goblinKW.jpeg  
  inflating: gambar/gambars/jokowi.png  
  inflating: gambar/gambars/koceng.jpeg  
  inflating: gambar/gambars/kucing.jpeg  
  inflating: gambar/gambars/kucingGweh.png  
  inflating: gambar/gambars/littlePoni.jpeg  
  inflating: gambar/gambars/lord.jpeg  
  inflating: gambar/gambars/statement.jpeg  
  inflating: gambar/gambars/ye.jpeg  
  inflating: gambar/gambars/yee.jpeg  
  inflating: gambar/gambars/yeee.jpeg  
  inflating: gambar/gambars/zebra.jpeg  


### Membaca dan menampilkan gambar
kalian bisa pake fungsi dasar OpenCV untuk nge-load gambar ke dalam memori

```
cv2.imread()
```

nampilinnya sebenernya bebas sih, misal kalian simpan di variable tinggal kalian run variablenya (masalahnya di collab agak aneh). atau umumnya pakai
```
cv2.imshow("judul image (bebas)", variable_image)
```
atau mau pake matplotlib juga gapapa
```
import matplotlib.pyplot as plt
plt.imshow(variable_image)
plt.axis('off') #buat ngilangin angka koordinat di pinggir (opsional)
plt.show()
```

### Converting gambar
misal kalian mau ubah warna warna, apalagi OpenCV itu pake format BGR. kalian bisa pake

```
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
```

"COLOR_BGR2RGB" ini kan bisa dibaca "BGR to RGB", nah masalahnya ada banyak kalau mau eksperimen. jadi ini kukasih ringkasan dari gemini

- cv2.COLOR_BGR2GRAY : Grayscale

basic yah, ganti warna ke grayscale

- cv2.COLOR_BGR2HSV : Hue, Saturation, Value

Di HSV, warna aslinya dikunci di variabel Hue. Kalau kamu mau ambil objek warna "Hijau" tapi hijaunya ada yang terang dan gelap karena bayangan, pakai HSV jauh lebih akurat daripada BGR

- cv2.COLOR_BGR2LAB : Lab*

Ruang warna ini dirancang menyerupai cara manusia melihat warna. Pemisahan antara kecerahan (Luminance) dan warna (a dan b) sangat tegas. Bagus banget buat membedakan objek yang warnanya mirip dengan latar belakang tapi punya tekstur atau pencahayaan berbeda

intinya kalian bebas mau pake apa

### Teknik Segmentasi

ada beberapa teknik yang bisa dipake, misal kalian ada cara sendiri boleh banget dipake. tapi yang ku dokumentasikan cuman beberapa aja

#### 1. Grayscale & Thresholding
ini kalau kalian ubah ke grayscale. abis itu kalian tinggal tentukan thresholdnya aja. buat thresholding bisa pake "cv2.threshold"

```
ret, mask = cv2.threshold(sumber_gambar, thresh, maxval, type)

- thresh: Nilai ambang batas (contoh: 127). Semua pixel di atas/bawah angka ini akan diproses.
- maxval: Nilai maksimum yang diberikan pada pixel yang lolos sensor (biasanya 255 untuk putih murni)
- type: peraturan pemotongan:
cv2.THRESH_BINARY: kalau > thresh, jadi putih. kalau < thresh, jadi hitam
cv2.THRESH_BINARY_INV: Kebalikan dari yang di atas (cocok kalau objeknya lebih gelap dari background)
```

Atau yang udah kita bahas, pake Otsu

```
ret, thresh_img = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

print(f"Nilai threshold optimal yang ditemukan Otsu: {ret}") # bisa kalian cek kayak gini, nilai optimalnya
```

Atau brutal kalian loop bae

```
eksperimen = [10, 40, 60, dst]
# terus kalian coba coba, ganti nilai thresh nya sampe yang menurut kalian ok
```
pake range dah tuh biar bikin video sekalian wkwk

*tips: kalau kalian beneran eksperimen nyoba nyoba threshold. pake "np.hstack" biar hasil hasilnya keliatan secara bersamaan

```
import numpy as np
ret1, batas50 = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY)
ret2, batas100 = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
ret3, batas150 = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
dst

# Gabungkan secara horizontal (Horizontal Stack)
perbandingan = np.hstack((batas50, batas100, batas150))

cv2.imshow("Perbandingan 50 vs 100 vs 150", perbandingan)
cv2.waitKey(0)
```

##### 1.5: Cleaning Up (Morphological Operations)
kayak yang udah kita bahas kemarin, threshold kemungkinan besar akan ninggalin noise, kalian bisa bersihin dengan morfologi

- erosi: mengikis pinggiran objek putih
- dilasi: menambah pixel pada pinggiran objek putih

```
# bikin kernel dulu bebas ukurannya, semakin besar angkanya, semakin kuat efek pengikisan/penebalannya
kernel = np.ones((5, 5), np.uint8)

# Erosi (Menghilangkan noise kecil)
img_erosion = cv2.erode(thresh_img, kernel, iterations=1)

# Dilasi (mempertebal)
img_dilation = cv2.dilate(img_erosion, kernel, iterations=1)
```
bisa kalian lakukan hanya salah satu, atau again, kalian bisa lakukan ber-urutan. bisa pake:
- Opening (Erosi, Dilatasi)
- Closing (Dilatasi, Erosi)

```
kernel = np.ones((3, 3), np.uint8)
opening = cv2.morphologyEx(thresh_otsu, cv2.MORPH_OPEN, kernel)
closing = cv2.morphologyEx(thresh_otsu, cv2.MORPH_CLOSE, kernel)
```

Selesai deh, foreground yang dihasilkan oleh eksperimen kalian, harus dipisahkan dan masukkan ke folder "result" ya

caranya bisa dengan "cv2.bitwise_and", dia cara kerjanya itu adalah dengan melihat mask (gambar hitam-putih hasil thresholding/morfologi) terus dibandingin dengan gambar asli, dan abis itu hanya mengambil warna dari gambar asli, tapi hanya di area yang masknya berwarna putih

kalo secara matematis kita pake "cv2.bitwise_and", karena warna Hitam di mask memiliki nilai 0, dan Putih memiliki nilai 1 (atau 255).
Apapun warna yang dikalikan dengan 0 (hitam) akan menjadi 0 (hitam).
Apapun warna yang dikalikan dengan 1 (putih) akan tetap menjadi warna aslinya


```
# fungsinya terima 3 parameter:
# 1. Input 1: Gambar asli
# 2. Input 2: Gambar asli
# 3. Mask: Hasil thresholding kalian.

foreground = cv2.bitwise_and(img_asli, img_asli, mask=mask_kalian)

os.makedirs(result)
filename = "hasil_objek.png"
cv2.imwrite(os.path.join('output', filename), foreground)
```

formatnya dalam bentuk png ya, jadi yang di mask hitam itu kita hilangkan (ya kalian tau lah kalau png itu backgroundnya ga ada kan)

kalian harus menambahkan Alpha Channel (Channel ke-4)

```
# kalian pisahin dulu channel B, G, R dari gambar asli
b, g, r = cv2.split(img_asli)

# pake mask hasil segmentasi sebagai channel ke-4 (Alpha)
# Masker: Putih (255) = Terlihat, Hitam (0) = Transparan
alpha = mask_hasil_morfologi

# Merge kembali menjadi BGRA
img_BGRA = cv2.merge([b, g, r, alpha])
cv2.imwrite('output/objek_transparan.png', img_BGRA)
'''

#### 2. Color Masking (HSV Space)
Teknik berikutnya yang bisa kalian gunakan (atau ngga juga gapapa) adalah HSV Space. ini better karena berbeda dengan RGB/BGR yang mencampur warna berdasarkan intensitas cahaya, HSV memisahkan komponen warna (Hue) dari kejenuhan (Saturation) dan kecerahannya (Value)

proses awalnya sama tapi bedanya kalian konversi ke HSV

Abis itu kalian tentuin batas bawah (lower) dan batas atas (upper) warna yang mau diambil (nanti ku kasih cheatsheet nya kok, tenang aja. tapi kalian tetep boleh kalau mau eksperimen)

```
# Nilai Hue di OpenCV adalah 0-179
# misal mau nyari ijo
lower_green = np.array([35, 50, 50]) # H, S, V
upper_green = np.array([85, 255, 255]) # sama
```

Dalam matematika warna, lingkaran Hue itu aslinya adalah 360°. Namun, karena OpenCV menggunakan tipe data uint8 (8-bit) yang kapasitas maksimumnya cuma 255, angka 360 nggak akan muat.

- Hue (H): OpenCV membagi nilai asli (0-360) menjadi dua. Jadi, $360 / 2 = 180$. Itulah kenapa rentangnya cuma 0 sampai 179.

- Saturation (S) & Value (V): Keduanya adalah persentase (0-100%). Untuk memaksimalkan kapasitas 8-bit, OpenCV memetakan 0-100% ini ke rentang 0 sampai 255.

ini cheatsheetnya ya:
| Warna  | Lower Bound (H, S, V) | Upper Bound (H, S, V) |
|--------|----------------------|----------------------|
| Orange | [5, 50, 50]          | [15, 255, 255]       |
| Yellow | [20, 50, 50]         | [35, 255, 255]       |
| Green  | [35, 50, 50]         | [85, 255, 255]       |
| Blue   | [100, 50, 50]        | [130, 255, 255]      |
| Red*   | [0, 50, 50]          | [10, 255, 255]       |


udah deh tinggal bikin masknya

```
mask = cv2.inRange(file, lower_green, upper_green)
```



#### 3. GrabCut
ini teknik terakhir yang ku dokumentasikan, intinya kalian ada ide sendiri, silahkan digunakan. ga wajib kok pake dokumentasiku

```
cv2.grabCut(img, mask, rect, bgdModel, fgdModel, iterCount, mode)

- img: Gambar asli kamu (BGR).
- mask: Ini "papan tulis" tempat GrabCut bekerja. Ukurannya harus sama dengan gambar asli.
- rect: Ini adalah kotak sakti kita. Formatnya (x, y, w, h).
- bgdModel & fgdModel: Ini hanya array kosong berukuran (1, 65) yang dibutuhkan OpenCV untuk menyimpan data statistik selama proses hitung. Abaikan saja isinya, yang penting harus dibuat aja
- iterCount: Berapa kali komputer harus mengulang proses pembersihan. Biasanya 5 sudah cukup bersih. Terlalu banyak malah bikin lemot.
- mode: bisa pake cv2.GC_INIT_WITH_RECT karena kita buat instruksi dengan kotak
```

contoh

```
# Buat masker kosong berukuran sama dengan gambar
mask = np.zeros(img.shape[:2], np.uint8)

# Definisikan model latar belakang & depan (digunakan internal oleh algoritma)
bgdModel = np.zeros((1, 65), np.float64)
fgdModel = np.zeros((1, 65), np.float64)

# Tentukan area objek (, y, w, h) ini angka ya
rect = (x, y, w, h)

# Jalankan GrabCut
cv2.grabCut(img, mask, rect, bgdModel, fgdModel, 5, cv2.GC_INIT_WITH_RECT)

# Ubah masker: pixel yang pasti/mungkin foreground (1 & 3) jadi 1, sisanya 0
mask2 = np.where((mask==2)|(mask==0), 0, 1).astype('uint8')

# Kalikan dengan gambar asli
output = img * mask2[:, :, np.newaxis]
```